# 48. Inference, Error Analysis와 결과 시각화

이 노트북은 segmentation 모델의 예측 결과를 시각화하고 실패 사례를 분석하는 방법을 다룹니다.

이번 노트북의 목표는 다음과 같습니다.

- logits에서 prediction map을 얻는 과정을 확인합니다.
- mask overlay와 error map을 시각화합니다.
- 실패 사례를 개선 실험으로 연결하는 기준을 정리합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(5)

## 48-1. 예제 ground truth와 prediction 만들기

In [ ]:
h, w, num_classes = 96, 96, 3
yy, xx = np.mgrid[:h, :w]

image = np.zeros((h, w, 3), dtype=np.float32) + 0.55
target = np.zeros((h, w), dtype=np.int64)

circle = (xx - 35) ** 2 + (yy - 46) ** 2 < 18 ** 2
rect = (xx > 54) & (xx < 82) & (yy > 30) & (yy < 74)
image[circle] = [0.9, 0.25, 0.22]
image[rect] = [0.2, 0.55, 0.9]
target[circle] = 1
target[rect] = 2

pred = target.copy()
pred[(xx - 35) ** 2 + (yy - 46) ** 2 > 15 ** 2] = np.where(target[(xx - 35) ** 2 + (yy - 46) ** 2 > 15 ** 2] == 1, 0, pred[(xx - 35) ** 2 + (yy - 46) ** 2 > 15 ** 2])
pred[(xx > 74) & (yy > 60)] = 0
pred[(xx > 45) & (xx < 58) & (yy > 28) & (yy < 42)] = 2

cmap = ListedColormap(["#8b95a1", "#e53935", "#2563eb"])

## 48-2. Prediction map과 overlay

In [ ]:
def colorize(mask):
    palette = np.array([
        [139, 149, 161],
        [229, 57, 53],
        [37, 99, 235],
    ], dtype=np.float32) / 255.0
    return palette[mask]


def overlay(image, mask, alpha=0.45):
    return np.clip((1 - alpha) * image + alpha * colorize(mask), 0, 1)


fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(target, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title("ground truth")
axes[2].imshow(pred, cmap=cmap, vmin=0, vmax=2)
axes[2].set_title("prediction")
axes[3].imshow(overlay(image, pred))
axes[3].set_title("overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 48-3. Error map

In [ ]:
error = pred != target

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(image)
axes[0].imshow(error, cmap="Reds", alpha=0.55)
axes[0].set_title("error overlay")
axes[1].imshow(error, cmap="gray")
axes[1].set_title("error map")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("error pixel ratio:", round(float(error.mean()), 4))

## 48-4. Class별 실패 비율

In [ ]:
for cls in range(num_classes):
    cls_area = target == cls
    if cls_area.sum() == 0:
        continue
    cls_error = (pred[cls_area] != cls).mean()
    print(f"class {cls} error ratio:", round(float(cls_error), 4))

## 48-5. 실패 사례를 실험으로 연결하기

| 관찰된 실패 | 가능한 원인 | 다음 실험 |
|---|---|---|
| 작은 객체를 놓침 | 해상도 부족, class imbalance | crop size 증가, loss weight 조정 |
| 경계가 거침 | decoder 표현력 부족, resize 문제 | decoder dim 조정, boundary-aware augmentation |
| 특정 class만 약함 | 데이터 부족, annotation 품질 | class sampling, label 검수 |
| 배경 false positive 많음 | threshold, hard negative 부족 | augmentation, negative sample 추가 |

결과 시각화는 단순 확인용이 아니라 다음 실험을 정하는 도구입니다.

## 48-6. 6장 전체 흐름 정리

```text
41. 프로젝트 개요
42. 데이터셋 구조와 annotation
43. Dataset과 DataLoader
44. train / validation loop
45. loss와 class imbalance
46. metric 계산
47. augmentation과 해상도 실험
48. inference와 error analysis
```

## 최종 정리

- 실제 segmentation 프로젝트는 모델 구조보다 데이터와 평가 구현에서 자주 흔들립니다.
- image-mask pair, nearest mask resize, ignore index, mIoU 계산을 먼저 안정화해야 합니다.
- 예측 결과는 overlay와 error map으로 확인하고, 실패 유형을 다음 실험으로 연결해야 합니다.
- 다음 단계로는 SegFormer fine-tuning, foundation model 활용, 또는 배포 최적화로 확장할 수 있습니다.